## Virtual Environment

Create and activate a virtual environment:

```bash
python -m venv irony
source venv/bin/activate  # On Windows: venv\Scripts\activate
```

or using conda
```
conda create -n irony python=3.14.3
conda activate irony
```
Install kernel

```bash
pip install --user ipykernel
python -m ipykernel install --user --name=irony
```

## Install Dependencies

```bash
pip install -r requirements.txt
```

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "allenai/OLMo-2-0425-1B-Instruct-GGUF"  # lightest one

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,   # use float32 if on CPU
    device_map="auto"            # auto-assigns to GPU if available
)

def check_model_loaded(model, tokenizer):
    test_input = "Hello, my name is"
    
    inputs = tokenizer(test_input, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=10)
    
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Input:  {test_input}")
    print(f"Output: {decoded}")
    print(f"Device: {model.device}")
    print(f"Dtype:  {model.dtype}")

check_model_loaded(model, tokenizer)


ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [3]:
import pandas as pd

files = {
    "condition1": "./conditions/Condition1(context_richness).csv",
    "condition2": "./conditions/Condition2(common_ground).csv",
    "condition3": "./conditions/Condition3(prompting_style).csv",
}

In [ ]:
def run_zero_shot(prompt: str, max_new_tokens: int = 258) -> str:
    # OLMo2 Instruct uses a chat template
    messages = [{"role": "user", "content": prompt}]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

        # If it's a BatchEncoding, unpack it; if it's already a tensor, keep it
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids["input_ids"]
    
    input_ids = input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    response = tokenizer.decode(
        output[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )
    return response.strip()

In [ ]:
import pandas as pd
from tqdm import tqdm

# Load and process all 3 files
files = {
    "condition1": "./conditions/Condition1(context_richness).csv",
    "condition2": "./conditions/Condition2(common_ground).csv",
    "condition3": "./conditions/Condition3(prompting_style).csv",
}

for name, path in files.items():
    print(f"\nProcessing {name}...")
    
    df = pd.read_csv(path)
    print(df.head())  # sanity check
    
    tqdm.pandas()
    df["model_output"] = df["full_prompt"].progress_apply(run_zero_shot)
    
    output_path = f"results_olmo2_zeroshot_{name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

print("\nAll done!")


Processing condition1...
      item_id context-level irony_label                 basic_utterance  \
0     C1_01_L           low      ironic  What a great start to the day.   
1  C1_01_L_NI           low  non-ironic  What a great start to the day.   
2     C1_01_H          high      ironic  What a great start to the day.   
3  C1_01_H_NI          high  non-ironic  What a great start to the day.   
4     C1_02_L           low      ironic      That went really smoothly.   

                                         full_prompt  
0  Is the following statement ironic? Answer with...  
1  Is the following statement ironic? Answer with...  
2  Is the following statement ironic? Answer with...  
3  Is the following statement ironic? Answer with...  
4  Is the following statement ironic? Answer with...  


100%|█████████████████████████████████████████| 120/120 [02:29<00:00,  1.25s/it]


Saved: results_olmo2_zeroshot_condition1.csv

Processing condition2...
      item_id cg_level irony_label                           base_utterance  \
0   C2_01_L_I      low      ironic          You are such a reliable friend.   
1  C2_01_L_NI      low  non-ironic          You are such a reliable friend.   
2   C2_01_H_I     high      ironic          You are such a reliable friend.   
3  C2_01_H_NI     high  non-ironic          You are such a reliable friend.   
4   C2_02_L_I      low      ironic  Thanks for being so punctual as always.   

                                         full_prompt  
0  Is the following statement ironic? Answer with...  
1  Is the following statement ironic? Answer with...  
2  Background (known to both): Sam took three day...  
3  Background (known to both): Sam dropped everyt...  
4  Is the following statement ironic? Answer with...  


100%|█████████████████████████████████████████| 119/119 [02:31<00:00,  1.27s/it]


Saved: results_olmo2_zeroshot_condition2.csv

Processing condition3...
      item_id prompt_level irony_label                 base_utterance  \
0   C3_01_L_I          low      ironic  That went exactly as planned.   
1  C3_01_L_NI          low  non-ironic  That went exactly as planned.   
2   C3_01_H_I         high      ironic  That went exactly as planned.   
3  C3_01_H_NI         high  non-ironic  That went exactly as planned.   
4   C3_02_L_I          low      ironic    I love how this turned out.   

                                         full_prompt  
0  Consider the following before answering:\n (1)...  
1  Consider the following before answering:\n (1)...  
2  Consider the following before answering:\n (1)...  
3  Consider the following before answering:\n (1)...  
4  Consider the following before answering:\n (1)...  


100%|█████████████████████████████████████████| 100/100 [02:10<00:00,  1.31s/it]

Saved: results_olmo2_zeroshot_condition3.csv

All done!


In [5]:
df.to_csv("results_olmo2_zeroshot.csv", index=False)
print("Done! Results saved.")

Done! Results saved.
